In [ ]:
from pathlib import Path
import platform

REPO = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "dpsim-fmi/CMakeLists.txt").is_file()
    ),
    None,
)
if REPO is None:
    raise RuntimeError("Start the kernel inside the DPsim repository.")
BUILD_ROOT = REPO / (
    "build-fmi" if platform.system() == "Linux" else "build-macos-check"
)
CASE = REPO / "dpsim-fmi/examples/inverter"
BUILD = BUILD_ROOT / "fmi-examples/inverter"
FMU_NAME = (
    "GridTiedInverterPortedV3.x86_64-linux.fmu"
    if platform.system() == "Linux"
    else "GridTiedInverterPortedV3.fmu"
)
FMU = CASE / "sources" / FMU_NAME
EXE = BUILD_ROOT / "dpsim-fmi/FMI_EMT_Ph3_Mixed"
BUILD.mkdir(parents=True, exist_ok=True)
if not (BUILD / "matlab-reference").exists():
    (BUILD / "matlab-reference").symlink_to(CASE / "matlab", target_is_directory=True)
import shutil, subprocess

run = BUILD / "dpsim-native-source"
run.mkdir(exist_ok=True)
shutil.copyfile(CASE / "sources/config.json", run / "config.json")
if not (run / "logs/fmi_mixed.csv").exists():
    result = subprocess.run(
        [
            str(EXE),
            str(FMU),
            "config.json",
        ],
        cwd=run,
        capture_output=True,
        text=True,
    )
    (run / "run.log").write_text(result.stdout + result.stderr)
    assert result.returncode == 0, result.stderr
for name in ["tightening.json", "startup_refinement.json", "refinement.json"]:
    shutil.copyfile(CASE / "matlab" / name, BUILD / name)

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = BUILD
BASES = {
    "voltage_V": 25000 * np.sqrt(2 / 3),
    "current_A": 500000 / 25000 * np.sqrt(2 / 3),
    "pu": 1.0,
    "frequency_Hz": 60.0,
    "angle_rad": 2 * np.pi,
}


def read_dpsim(folder):
    d = pd.read_csv(ROOT / folder / "logs/fmi_mixed.csv", skipinitialspace=True)
    c = json.loads((ROOT / folder / "config.json").read_text())
    # V2's first/third PLL measurement labels were reversed; V3 corrects names.
    if c.get("fmu_variant", "V2") == "V2":
        d = d.rename(
            columns={"pll_magnitude": "pll_frequency", "pll_frequency": "pll_magnitude"}
        )
    return d


def compare(folder, reference, label, stop_time=None):
    d = read_dpsim(folder)
    reference_path = ROOT / "matlab-reference" / reference
    if not reference_path.is_file():
        pairs = {}
        for k, p in enumerate("abc"):
            pairs[f"v_pcc_{p}"] = (d[f"pcc_voltage_{k}"], None, BASES["voltage_V"])
            pairs[f"i_grid_{p}"] = (-d[f"line_current_{k}"], None, BASES["current_A"])
            pairs[f"i_converter_{p}"] = (
                d[f"converter_current_{k}"],
                None,
                BASES["current_A"],
            )
            pairs[f"i_load_{p}"] = (d[f"load_current_{k}"], None, BASES["current_A"])
        for s in [
            "v_d_pu",
            "v_q_pu",
            "pi_d",
            "pi_q",
            "pll_frequency",
            "pll_theta",
            "pll_magnitude",
        ]:
            base = (
                BASES["frequency_Hz"]
                if s == "pll_frequency"
                else BASES["angle_rad"] if s == "pll_theta" else 1.0
            )
            pairs[s] = (d[s], None, base)
        metric_columns = [
            "case",
            "signal",
            "base",
            "max_abs",
            "rms_abs",
            "max_normalized",
            "rms_normalized",
            "raw_max_abs",
            "pass_0p1pct",
        ]
        metrics = pd.DataFrame(columns=metric_columns)
        diagnostics = dict(
            samples=len(d),
            start_time_s=float(d.time.iloc[0]),
            stop_time_s=float(d.time.iloc[-1]),
            max_newton_residual=float(d.newton_residual.max()),
            max_iterations=float(d.newton_iterations.max()),
            kcl_max_A=max(
                float(
                    np.max(
                        abs(
                            d[f"line_current_{k}"]
                            + d[f"converter_current_{k}"]
                            + d[f"load_current_{k}"]
                        )
                    )
                )
                for k in range(3)
            ),
            reference_available=False,
        )
        (ROOT / f"diagnostics_{label}.json").write_text(
            json.dumps(diagnostics, indent=2)
        )
        print(f"Reference not found: {reference_path}")
        print(
            "Loaded and validated the DPsim Linux result without a MATLAB comparison."
        )
        print(diagnostics)
        return metrics, d, None, pairs
    r = pd.read_csv(reference_path)
    if stop_time is not None:
        d = d[d.time <= stop_time + 1e-12].reset_index(drop=True)
        r = r[r.time <= stop_time + 1e-12].reset_index(drop=True)
    # Only use physically coincident output times, without trace shifts.
    t = d.time.to_numpy()
    rt = r.time.to_numpy()
    idx = np.searchsorted(rt, t)
    idx = np.clip(idx, 0, len(rt) - 1)
    prev = np.maximum(idx - 1, 0)
    idx = np.where(abs(rt[prev] - t) < abs(rt[idx] - t), prev, idx)
    assert np.max(abs(rt[idx] - t)) < 1e-11, "Unaligned time series"
    r = r.iloc[idx].reset_index(drop=True)
    pairs = {}
    for k, p in enumerate("abc"):
        pairs[f"v_pcc_{p}"] = (
            d[f"pcc_voltage_{k}"],
            r[f"v_pcc_{p}"],
            BASES["voltage_V"],
        )
        pairs[f"i_grid_{p}"] = (
            -d[f"line_current_{k}"],
            r[f"i_grid_{p}"],
            BASES["current_A"],
        )
        pairs[f"i_converter_{p}"] = (
            d[f"converter_current_{k}"],
            r[f"i_grid_{p}"] - r[f"v_pcc_{p}"] / 12500,
            BASES["current_A"],
        )
        pairs[f"i_load_{p}"] = (
            d[f"load_current_{k}"],
            r[f"v_pcc_{p}"] / 12500,
            BASES["current_A"],
        )
    for s in [
        "v_d_pu",
        "v_q_pu",
        "pi_d",
        "pi_q",
        "pll_frequency",
        "pll_theta",
        "pll_magnitude",
    ]:
        base = (
            BASES["frequency_Hz"]
            if s == "pll_frequency"
            else BASES["angle_rad"] if s == "pll_theta" else 1.0
        )
        pairs[s] = (d[s], r[s], base)
    rows = []
    for s, (actual, expected, base) in pairs.items():
        e = np.asarray(actual) - np.asarray(expected)
        rawmax = float(np.max(abs(e)))
        if s == "pll_theta":
            e = (e + np.pi) % (2 * np.pi) - np.pi
        peak = float(np.max(abs(e)))
        rms = float(np.sqrt(np.mean(e * e)))
        rows.append(
            dict(
                case=label,
                signal=s,
                base=base,
                max_abs=peak,
                rms_abs=rms,
                max_normalized=peak / base,
                rms_normalized=rms / base,
                raw_max_abs=rawmax,
                pass_0p1pct=peak / base < 0.001 and rms / base < 0.001,
            )
        )
    metrics = pd.DataFrame(rows)
    metrics.to_csv(ROOT / f"metrics_{label}.csv", index=False)
    diagnostics = dict(
        samples=len(d),
        start_time_s=float(d.time.iloc[0]),
        stop_time_s=float(d.time.iloc[-1]),
        max_newton_residual=float(d.newton_residual.max()),
        max_iterations=float(d.newton_iterations.max()),
        kcl_max_A=max(
            float(
                np.max(
                    abs(
                        d[f"line_current_{k}"]
                        + d[f"converter_current_{k}"]
                        + d[f"load_current_{k}"]
                    )
                )
            )
            for k in range(3)
        ),
        all_pass=bool(metrics.pass_0p1pct.all()),
    )
    (ROOT / f"diagnostics_{label}.json").write_text(json.dumps(diagnostics, indent=2))
    print(
        metrics[
            ["signal", "max_abs", "rms_abs", "max_normalized", "pass_0p1pct"]
        ].to_string(index=False)
    )
    print(diagnostics)
    return metrics, d, r, pairs

# Converter FMU connected to native DPsim EMT three-phase components

The official MathWorks grid-tied inverter is imported as FMI 3.0 Model Exchange and connected in the DPsim MNA system to a native three-phase voltage source, line resistor, line inductor and shunt resistor load. The FMU retains the IGBT/diode switching plant, transformer, filter, original loads, PLL, two PI loops, PWM and breaker. Its source boundary is at 25 kV; the inverter side is 380 V.

**Topology:** native grid → native R → native L → PCC → converter FMU, with native load at the PCC. Breaker closure inside the original converter model remains at 0.15 s. Simulation covers 0–0.2 s.

The user accepted the measured error margin for this mixed-component demonstration. The initial 0.1% normalized maximum/RMS target is retained as a diagnostic column and is not represented as universally passed. Engineering current base is 16.3299 A peak (500 kVA, 25 kV); the actual converter-current peak is only about 2.53 A. No traces are shifted and no startup interval is discarded.


In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = BUILD
sys.path.insert(0, str(ROOT))

cases = {
    "native_1us": compare("dpsim-native-source", "reference_trap_1us.csv", "native_1us")
}
quarter_reference = ROOT / "matlab-reference/reference_trap_0p25us.csv"
if quarter_reference.is_file():
    cases["native_1us_vs_quarter_reference"] = compare(
        "dpsim-native-source", quarter_reference.name, "native_1us_vs_quarter_reference"
    )
assert "native_1us" in cases, "Final native-source run must complete"
metric_frames = [item[0] for item in cases.values() if not item[0].empty]
all_metrics = (
    pd.concat(metric_frames, ignore_index=True) if metric_frames else pd.DataFrame()
)
all_metrics.to_csv(ROOT / "metrics_all.csv", index=False)
if not all_metrics.empty:
    display(
        all_metrics[
            [
                "case",
                "signal",
                "base",
                "max_abs",
                "rms_abs",
                "max_normalized",
                "rms_normalized",
                "pass_0p1pct",
            ]
        ]
    )
else:
    print("No MATLAB reference CSV is available; reference error metrics were skipped.")

In [ ]:
label = "native_1us"
metrics, d, r, pairs = cases[label]
fig, axes = plt.subplots(4, 1, figsize=(11, 10))
for ax, (signal, scale, unit, window) in zip(
    axes,
    [
        ("v_pcc_a", 1e-3, "kV", (0, 0.2)),
        ("i_converter_a", 1, "A", (0.145, 0.16)),
        ("pi_q", 1, "pu", (0, 0.2)),
        ("pll_frequency", 1, "Hz", (0, 0.2)),
    ],
):
    actual, expected, base = pairs[signal]
    # Plot decimation only; every saved sample is used for metrics.
    stride = 1 if window[0] > 0.1 else 20
    if expected is not None:
        ax.plot(
            d.time.iloc[::stride],
            np.asarray(expected)[::stride] * scale,
            label="Independent Simulink trapezoid",
            lw=1,
        )
    ax.plot(
        d.time.iloc[::stride],
        np.asarray(actual)[::stride] * scale,
        "--",
        label="DPsim Linux FMU",
        lw=1,
    )
    ax.set(xlim=window, ylabel=f"{signal} [{unit}]")
    ax.grid(alpha=0.25)
    ax.axvline(0.15, color="k", alpha=0.35, lw=0.8)
axes[0].legend()
axes[-1].set_xlabel("Time [s]")
fig.suptitle(label)
fig.tight_layout()
fig.savefig(ROOT / "mixed_comparison.png", dpi=160)
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
reference_cases = 0
for label, (metrics, d, r, pairs) in cases.items():
    if r is None:
        continue
    reference_cases += 1
    for ax, signal in zip(axes, ["i_converter_a", "pi_q", "pll_frequency"]):
        actual, expected, base = pairs[signal]
        error = 100 * (np.asarray(actual) - np.asarray(expected)) / base
        ax.plot(d.time.iloc[::20], error[::20], label=label, lw=0.7)
        ax.set_ylabel(signal + " error [% base]")
        ax.grid(alpha=0.25)
if reference_cases == 0:
    for ax in axes:
        ax.text(
            0.5,
            0.5,
            "MATLAB reference CSV not available",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )
for ax in axes:
    ax.axvline(0.15, color="k", alpha=0.35, lw=0.8)
    if reference_cases:
        ax.legend()
axes[-1].set_xlabel("Time [s]")
fig.tight_layout()
fig.savefig(ROOT / "mixed_errors.png", dpi=160)
plt.show()

In [ ]:
diagnostics = {
    label: json.loads((ROOT / f"diagnostics_{label}.json").read_text())
    for label in cases
}
config = json.loads((ROOT / "dpsim-native-source/config.json").read_text())
expected_samples = (
    round(config["stop_time_s"] / config["step_s"] / config.get("log_downsampling", 1))
    + 1
)
for label, item in diagnostics.items():
    assert item["samples"] >= expected_samples and item["max_newton_residual"] <= 1
    assert item["kcl_max_A"] < 1e-5
print(json.dumps(diagnostics, indent=2))
tight = json.loads((ROOT / "tightening.json").read_text())
print(
    "Maximum physical/control difference after 10x Newton tightening:",
    max(tight.values()),
)
assert max(tight.values()) == 0

## Reproduction and limits

Build the `FMI_EMT_Ph3_Mixed` target in the repository build directory. The first cell runs the baseline if needed; the validation section below creates each disturbance configuration and runs missing cases. Final FMU, matching XML and source model are in `sources/`; physical MATLAB reference models/results are in `matlab/`. Generated files stay under build.

The converter retains the original switching devices, PLL, PI loops, PWM, transformer/filter and breaker. MATLAB references simulate the independent physical Simscape network and do not invoke the FMU. This is not a hand-derived converter ODE. It is a grid-tied inverter benchmark, not a passed PV/MPPT benchmark.

The provisional 0.1% engineering-base threshold remains visible for every signal. Some converter/controller signals exceed it; acceptance of those errors does not establish a universal importer accuracy guarantee. Full-trajectory metrics include startup and disturbance peaks. Angle errors are wrapped, with raw maxima also retained.

The practical requested network step is **50 µs**. Current switched-export results use **1 µs** because the FMU schedules internal events at that interval. Direct 50/25 µs attempts below test the actual limitation, without skipping events or retiming controls. Smaller MATLAB steps estimate reference uncertainty; they do not prove FMU convergence. Existing baseline quarter-step comparisons include a short startup discrepancy of approximately 325 V (1.59% voltage base).

All work remains local under the SSD development directory, on `feature/FMU_import`, uncommitted and unpushed. No Modelica models or tools are used.

## Native switched loads and disturbed grid

Five separate 0–0.2 s runs exercise balanced additional load (25 kΩ per phase), unbalanced load (25/12.5/50 kΩ), a 20% voltage sag from 0.17 to 0.19 s, a +10° phase jump, and a continuous-phase frequency change from 60 to 59 Hz at 0.17 s. The original internal breaker still closes at 0.15 s. The added loads are native DPsim EMT components. Reference switches use the same 1 mΩ closed and 1 TΩ open resistance.

The independent reference preserves the original grid source, its internal R/L and zero-current initialization. Grid disturbances use stateless series voltage corrections; they introduce no extra electrical storage. Both controller subsystem XMLs are byte-identical to the original physical reference (checked below). The supplied MATLAB models can be opened and simulated directly with the supplied data file and physical-signal library on the path.

Event parameters in MATLAB use `170000*1e-6` and `190000*1e-6`, matching integer-grid event times; there are no shifted comparison traces. Control outputs are held using previous samples only when a finer reference has a different output grid. Metrics use every coincident sample, with both full and post-disturbance windows.

Engineering bases: 25 kV line-to-line RMS and 500 kVA imply 20,412.4145 V phase peak and 16.32993 A phase peak; controller pu outputs use 1 pu, PLL frequency 60 Hz and angle 2π rad. These declared bases are unchanged across cases.

In [ ]:
VALIDATION = BUILD_ROOT / "fmi-validation"
VALIDATION.mkdir(parents=True, exist_ok=True)
settings = json.loads((CASE / "sources/validation_cases.json").read_text())
base_config = json.loads((CASE / "sources/config.json").read_text())
exe = EXE
fmu = FMU
for name, overrides in settings.items():
    for variant in ["", "_tight"]:
        folder = VALIDATION / (name + variant)
        folder.mkdir(exist_ok=True)
        config = {**base_config, **overrides}
        if variant:
            config["tolerance_scale"] = 0.1
        config_path = folder / "config.json"
        if config_path.exists():
            existing = json.loads(config_path.read_text())
            for legacy in [
                "voltage_inputs",
                "current_outputs",
                "current_sign_to_passive",
            ]:
                existing.pop(legacy, None)
            assert existing == config
        else:
            config_path.write_text(json.dumps(config, indent=2) + "\n")
        if (
            not (folder / "logs/fmi_mixed.csv").exists()
            or "completed:" not in (folder / "run.log").read_text()
        ):
            result = subprocess.run(
                [str(exe), str(fmu), "config.json"],
                cwd=folder,
                capture_output=True,
                text=True,
            )
            (folder / "run.log").write_text(result.stdout + result.stderr)
            assert result.returncode == 0, result.stderr
    for reference in (CASE / "matlab/validation" / name).glob("reference*.csv"):
        shutil.copyfile(reference, VALIDATION / name / reference.name)
checks = json.loads((CASE / "matlab/validation/controller_integrity.json").read_text())
import zipfile

with zipfile.ZipFile(CASE / "matlab/GridTiedInverterMixedReferenceTrap.slx") as archive:
    original = {name: archive.read(name) for name in next(iter(checks.values()))}
for case_name, files in checks.items():
    models = list((CASE / "matlab/validation" / case_name).glob("*.slx"))
    assert len(models) == 1
    with zipfile.ZipFile(models[0]) as archive:
        assert all(archive.read(name) == original[name] for name in files)
description_file = (
    CASE
    / "sources"
    / (
        "modelDescription.x86_64-linux.xml"
        if platform.system() == "Linux"
        else "modelDescription.xml"
    )
)
with zipfile.ZipFile(fmu) as archive:
    assert archive.read("modelDescription.xml") == description_file.read_bytes()
assert all(all(files.values()) for files in checks.values())
print("All five cases and tightened runs completed; controller integrity:", checks)

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, json

B = VALIDATION
NAMES = [
    "balanced_load",
    "unbalanced_load",
    "voltage_sag",
    "phase_jump",
    "frequency_change",
]
VBASE = 25000 * np.sqrt(2 / 3)
IBASE = 500000 / 25000 * np.sqrt(2 / 3)


def compare_case(name, reference_name="reference_trap_1us.csv"):
    folder = B / name
    c = json.loads((folder / "config.json").read_text())
    d = pd.read_csv(folder / "logs/fmi_mixed.csv", skipinitialspace=True)
    r = pd.read_csv(folder / reference_name)
    assert len(d) == 200001
    times = d.time.to_numpy()
    rt = r.time.to_numpy()
    index = np.clip(np.searchsorted(rt, times), 0, len(rt) - 1)
    previous = np.maximum(index - 1, 0)
    index = np.where(
        abs(rt[previous] - times) < abs(rt[index] - times), previous, index
    )
    assert np.max(abs(rt[index] - times)) < 1e-11, "Reference times do not coincide"
    r = r.iloc[index].reset_index(drop=True)
    pairs = {}
    kcl = []
    for k, p in enumerate("abc"):
        v = r["v_pcc_" + p].to_numpy()
        load = v / c["load_R_ohm"]
        extra = np.zeros(len(d))
        if "step_load_R_ohm_abc" in c:
            resist = c["step_load_R_ohm_abc"][k] + np.where(
                r.time >= c["load_step_time_s"] - 1e-12, 0.001, 1e12
            )
            extra = v / resist
            pairs["i_switched_" + p] = (d[f"step_load_current_{k}"], extra, IBASE)
        pairs["v_pcc_" + p] = (d[f"pcc_voltage_{k}"], v, VBASE)
        pairs["i_grid_" + p] = (-d[f"line_current_{k}"], r["i_grid_" + p], IBASE)
        pairs["i_converter_" + p] = (
            d[f"converter_current_{k}"],
            r["i_grid_" + p] - load - extra,
            IBASE,
        )
        pairs["i_load_" + p] = (d[f"load_current_{k}"], load, IBASE)
        residual = (
            d[f"line_current_{k}"]
            + d[f"converter_current_{k}"]
            + d[f"load_current_{k}"]
        )
        if "step_load_R_ohm_abc" in c:
            residual = residual + d[f"step_load_current_{k}"]
        kcl.append(float(np.max(abs(residual))))
    for s in [
        "v_d_pu",
        "v_q_pu",
        "pi_d",
        "pi_q",
        "pll_frequency",
        "pll_theta",
        "pll_magnitude",
    ]:
        pairs[s] = (
            d[s],
            r[s],
            60 if s == "pll_frequency" else 2 * np.pi if s == "pll_theta" else 1,
        )
    rows = []
    for signal, (a, b, base) in pairs.items():
        error = np.asarray(a) - np.asarray(b)
        raw_error = error.copy()
        if signal == "pll_theta":
            error = (error + np.pi) % (2 * np.pi) - np.pi
        for window, mask in [
            ("full", np.ones(len(d), dtype=bool)),
            (
                "post_disturbance",
                d.time
                >= c.get("load_step_time_s", c.get("grid_event_time_s", 0.17)) - 1e-12,
            ),
        ]:
            e = error[mask]
            peak = float(max(abs(e)))
            rms = float(np.sqrt(np.mean(e * e)))
            rows.append(
                dict(
                    case=name,
                    reference=reference_name,
                    window=window,
                    signal=signal,
                    base=base,
                    raw_max_abs=float(np.max(abs(raw_error[mask]))),
                    max_abs=peak,
                    rms_abs=rms,
                    max_normalized=peak / base,
                    rms_normalized=rms / base,
                    pass_0p1pct=peak / base < 0.001 and rms / base < 0.001,
                )
            )
    metrics = pd.DataFrame(rows)
    metrics.to_csv(
        folder
        / (
            "metrics.csv"
            if reference_name == "reference_trap_1us.csv"
            else "metrics_vs_" + reference_name
        ),
        index=False,
    )
    diagnostics = {
        "samples": len(d),
        "kcl_max_A": max(kcl),
        "newton_scaled_residual_max": float(d.newton_residual.max()),
        "newton_iterations_max": float(d.newton_iterations.max()),
    }
    tight = B / (name + "_tight") / "logs/fmi_mixed.csv"
    if tight.exists() and "completed:" in (tight.parent.parent / "run.log").read_text():
        t = pd.read_csv(tight, skipinitialspace=True)
        cols = [
            x
            for x in d.columns
            if x in t and x not in ["newton_iterations", "newton_residual"]
        ]
        diagnostics["tightening_max_difference"] = float(
            np.max(abs(d[cols].to_numpy() - t[cols].to_numpy()))
        )
    (
        folder
        / (
            "diagnostics.json"
            if reference_name == "reference_trap_1us.csv"
            else "diagnostics_vs_fine_reference.json"
        )
    ).write_text(json.dumps(diagnostics, indent=2) + "\n")
    print(
        name,
        diagnostics,
        metrics[metrics.window == "full"]
        .groupby(metrics.signal.str.startswith("i_"))[
            ["max_normalized", "rms_normalized"]
        ]
        .max()
        .to_dict(),
        flush=True,
    )
    return metrics, d, r, pairs

In [ ]:
validation_results = {}
for name in settings:
    reference_path = VALIDATION / name / "reference_trap_1us.csv"
    if reference_path.is_file():
        validation_results[name] = compare_case(name)
    else:
        print(f"Skipping MATLAB comparison for {name}: {reference_path} is unavailable")
if validation_results:
    validation_metrics = pd.concat(
        [x[0] for x in validation_results.values()], ignore_index=True
    )
    validation_metrics.to_csv(VALIDATION / "metrics_all_scenarios.csv", index=False)
    summary = validation_metrics.query("window == 'full'").copy()
    summary["max_pct_base"] = 100 * summary.max_normalized
    summary["rms_pct_base"] = 100 * summary.rms_normalized
    display(
        summary[
            [
                "case",
                "signal",
                "base",
                "max_abs",
                "rms_abs",
                "max_pct_base",
                "rms_pct_base",
                "pass_0p1pct",
            ]
        ]
    )
    for name in validation_results:
        diagnostic = json.loads((VALIDATION / name / "diagnostics.json").read_text())
        assert diagnostic["kcl_max_A"] < 1e-5
        assert diagnostic["newton_scaled_residual_max"] <= 1
        assert diagnostic["tightening_max_difference"] == 0
    print(
        "KCL, Newton convergence and tolerance checks passed for all cases with references."
    )
else:
    validation_metrics = pd.DataFrame()
    print(
        "No MATLAB validation CSVs are present; scenario comparison plots are skipped."
    )

In [ ]:
fig, axes = plt.subplots(5, 3, figsize=(15, 17))
for row, (name, (metrics, d, r, pairs)) in enumerate(validation_results.items()):
    for col, signal in enumerate(["v_pcc_a", "i_converter_a", "pi_q"]):
        ax = axes[row, col]
        actual, expected, base = pairs[signal]
        ax.plot(d.time.iloc[::20], np.asarray(expected)[::20], label="MATLAB", lw=0.8)
        ax.plot(
            d.time.iloc[::20],
            np.asarray(actual)[::20],
            "--",
            label="DPsim + FMU",
            lw=0.8,
        )
        ax.set(title=name + " / " + signal, xlim=(0.165, 0.2))
        ax.grid(alpha=0.25)
        if row == 0:
            ax.legend()
for ax in axes[-1]:
    ax.set_xlabel("Time [s]")
fig.tight_layout()
fig.savefig(VALIDATION / "all_scenarios.png", dpi=160)
plt.show()

## Timestep capability and reference uncertainty

A coarser requested timestep must be supported by the actual coupled solver, not inferred from a decimated plot. `NonlinearSSN` has two generic multirate policies. `rollback_newton` replays the complete interval and retains converged implicit network coupling. `single_pass_explicit` advances a non-rollback FMU once after the network solve, retains every internal event, and reports the resulting lagged-current coupling error in amperes. The policy is independent of this converter's state names and equations.

This MathWorks R2025a Model Exchange FMU advertises rollback, but identical replay changes its hidden switched trajectory. The importer detects and rejects that behavior. The executed 50/25 µs runs therefore use the explicit generic policy and compare every coincident sample with the validated 1 µs DPsim import. Full-window metrics retain the zero-current startup transient; a second window begins at 10 ms so sustained converter/network behavior is visible without deleting the startup samples.

In [ ]:
reference = pd.read_csv(
    BUILD / "dpsim-native-source/logs/fmi_mixed.csv", skipinitialspace=True
).set_index("time")
signal_bases = {
    **{f"converter_current_{p}": BASES["current_A"] for p in range(3)},
    **{f"pcc_voltage_{p}": BASES["voltage_V"] for p in range(3)},
    "pi_d": 1.0,
    "pi_q": 1.0,
    "pll_frequency": 60.0,
    "pll_magnitude": 1.0,
    "pll_theta": 2 * np.pi,
    "v_d_pu": 1.0,
    "v_q_pu": 1.0,
}
multirate = []
runs = {}
for step in [5e-5, 2.5e-5]:
    label = f"multirate-{step*1e6:g}us"
    folder = BUILD / label
    folder.mkdir(exist_ok=True)
    config = {**base_config, "step_s": step, "multirate_mode": "single_pass_explicit"}
    for legacy in ["voltage_inputs", "current_outputs", "current_sign_to_passive"]:
        config.pop(legacy, None)
    (folder / "config.json").write_text(json.dumps(config, indent=2) + "\n")
    if not (folder / "logs/fmi_mixed.csv").exists():
        result = subprocess.run(
            [str(exe), str(fmu), "config.json"],
            cwd=folder,
            capture_output=True,
            text=True,
        )
        (folder / "run.log").write_text(result.stdout + result.stderr)
        assert result.returncode == 0, result.stdout + result.stderr
    d = pd.read_csv(folder / "logs/fmi_mixed.csv", skipinitialspace=True)
    runs[label] = d
    r = reference.loc[np.round(d.time.to_numpy(), 9)]
    for window, start in [("full", 0.0), ("after_10ms", 0.01)]:
        keep = d.time.to_numpy() >= start
        for signal, base in signal_bases.items():
            error = d[signal].to_numpy() - r[signal].to_numpy()
            if signal == "pll_theta":
                error = (error + np.pi) % (2 * np.pi) - np.pi
            error = error[keep]
            multirate.append(
                {
                    "run": label,
                    "window": window,
                    "signal": signal,
                    "base": base,
                    "max_abs": np.max(np.abs(error)),
                    "rms": np.sqrt(np.mean(error**2)),
                    "max_percent_base": 100 * np.max(np.abs(error)) / base,
                    "rms_percent_base": 100 * np.sqrt(np.mean(error**2)) / base,
                }
            )
    coupling = d.fmi_coupling_error_A.iloc[1:]
    print(
        label,
        "coupling max/RMS [A]",
        coupling.max(),
        np.sqrt(np.mean(coupling**2)),
        "internal steps",
        d.fmi_internal_steps.iloc[1:].min(),
        d.fmi_internal_steps.iloc[1:].max(),
    )
multirate_metrics = pd.DataFrame(multirate)
multirate_metrics.to_csv(BUILD / "multirate_metrics.csv", index=False)
display(
    multirate_metrics.query("window == 'after_10ms'").sort_values(
        ["run", "max_percent_base"], ascending=[True, False]
    )
)
fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
for label, d in runs.items():
    r = reference.loc[np.round(d.time.to_numpy(), 9)]
    for ax, signal in zip(axes, ["pcc_voltage_0", "converter_current_0", "pi_q"]):
        ax.plot(
            d.time,
            100 * (d[signal].to_numpy() - r[signal].to_numpy()) / signal_bases[signal],
            label=label,
            lw=0.7,
        )
for ax, signal in zip(axes, ["PCC voltage A", "converter current A", "PI q"]):
    ax.set_ylabel(signal + " error [% base]")
    ax.grid(alpha=0.25)
    ax.legend()
axes[-1].set_xlabel("Time [s]")
fig.tight_layout()
fig.savefig(BUILD / "multirate_comparison.png", dpi=160)
plt.show()

## Interpretation

The generic rollback scheduler passes its independent sampled-event recurrence test and retains converged implicit coupling when FMI state restoration is reproducible. The converter's advertised rollback is not reproducible, so its 50/25 µs results use the explicitly selected single-pass policy. After 10 ms, 50 µs current and PCC-voltage errors remain below 0.1% of their declared engineering bases; some controller signals exceed 0.1% and remain reported. Halving the network step reduces the largest controller difference.

The very large first-sample PCC-voltage differences are the zero-current initialization exciting the native line at a coarse trapezoidal step. They are retained in the full-window table. A loaded operating-point initialization is therefore the next converter milestone. The single-pass result is a practical multirate import with measured coupling error, not a claim of converged implicit coupling for an FMU whose rollback is invalid.

## Regenerating the independent MATLAB results

The following optional cell copies the final physical models into build and simulates them there. It writes new CSVs separately, leaving the supplied results intact. To regenerate finer physical references, change `h` explicitly; preserve original `Ts`, `Tsc`, gains and control blocks. This does not run the imported FMU.

In [ ]:
RUN_MATLAB = (
    False  # Saved executed references are supplied; set True to regenerate all five.
)
MATLAB_DRIVER = "root='/Volumes/SSDMatthias/development/dpsim';\ncaseFolder=fullfile(root,'dpsim-fmi/examples/inverter');\nout=fullfile(root,'build-macos-check/fmi-validation/regenerated-matlab');\nif ~isfolder(out),mkdir(out);end\ncd(out);addpath(fullfile(caseFolder,'sources'),fullfile(caseFolder,'matlab/validation'));\nSimulink.fileGenControl('set','CacheFolder',fullfile(out,'cache'),'CodeGenFolder',fullfile(out,'codegen'),'createDir',true);\nmaxNumCompThreads(1);\nnames={'balanced_load','unbalanced_load','voltage_sag','phase_jump','frequency_change'};\nlabels={'time','i_grid_a','v_pcc_a','i_grid_b','v_pcc_b','i_grid_c','v_pcc_c','v_d_pu','v_q_pu','pi_d','pi_q','pll_frequency','pll_theta','pll_magnitude'};\nfor k=1:numel(names)\n  models=dir(fullfile(caseFolder,'matlab/validation',names{k},'*.slx'));assert(numel(models)==1);\n  localModel=fullfile(out,models(1).name);copyfile(fullfile(models(1).folder,models(1).name),localModel);\n  load_system(localModel);[~,model]=fileparts(localModel);\n  h=1e-6;set_param(model,'FixedStep',num2str(h,17),'StopTime','.2');\n  sc=find_system(model,'SearchDepth',1,'LookUnderMasks','all','RegExp','on','Name','Solver.*Configuration');\n  set_param(sc{1},'LocalSolverSampleTime',num2str(h,17));\n  r=sim(model,'ReturnWorkspaceOutputs','on','SaveOutput','on','OutputSaveName','yout','SaveFormat','Dataset','LimitDataPoints','off');\n  t=(0:round(.2/h))'*h;data=t;assert(r.yout.numElements==13);\n  for j=1:r.yout.numElements\n    signal=r.yout.getElement(j);method='linear';if j>=7,method='previous';end\n    data(:,end+1)=interp1(signal.Values.Time,signal.Values.Data,t,method,'extrap');\n  end\n  writetable(array2table(data,'VariableNames',labels),fullfile(out,[names{k} '_reference_trap_1us.csv']));\n  close_system(model,0);\nend\n"
MATLAB_DRIVER = MATLAB_DRIVER.replace(
    "/Volumes/SSDMatthias/development/dpsim", str(REPO)
).replace("build-macos-check", BUILD_ROOT.name)
if RUN_MATLAB:
    driver = VALIDATION / "regenerate_matlab_references.m"
    driver.write_text(MATLAB_DRIVER)
    command = "run('" + str(driver).replace("'", "''") + "')"
    with (VALIDATION / "matlab-regeneration.log").open("w") as log:
        result = subprocess.run(
            [
                str(
                    Path("/usr/local/MATLAB/R2025a/bin/matlab")
                    if platform.system() == "Linux"
                    else Path("/Volumes/SSDMatthias/MATLAB_R2025a.app/bin/matlab")
                ),
                "-batch",
                command,
            ],
            cwd=VALIDATION,
            stdout=log,
            stderr=subprocess.STDOUT,
        )
    assert result.returncode == 0, "See matlab-regeneration.log"
else:
    print(
        "Using supplied independent MATLAB results; regeneration commands are retained in this cell."
    )